# TimeFuse deployment-faithful study (ETTh1 / ETTh2)
Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all. Takes about 30-45 minutes. At the end it downloads `timefuse_results.zip` (a few hundred KB). Send that zip back.

All paths mirror the original run so results are directly comparable.

In [ ]:
!nvidia-smi -L
!mkdir -p /home/claude/ext /home/claude/tfdata /home/claude/tf_work
!cd /home/claude/ext && git clone -q https://github.com/ZhiningLiu1998/TimeFuse.git
!cd /home/claude/ext/TimeFuse && git log -1 --format=%H && sed -i 's/np\.Inf/np.inf/g' utils/tools.py
!pip -q install reformer_pytorch PyWavelets statsmodels pyarrow
!for f in ETTh1 ETTh2; do curl -sSL -o /home/claude/tfdata/$f.csv https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/$f.csv; done; wc -l /home/claude/tfdata/*.csv

In [ ]:
%%writefile /home/claude/tf_work/train_base.py
import sys, os, time, json
sys.path.insert(0, '/home/claude/ext/TimeFuse'); os.chdir('/home/claude/ext/TimeFuse')
import torch, numpy as np, random
from load_configs import get_forecast_exp_args
from exp.exp_fuse_forecasting import Exp_Fuse_Forecasting
torch.set_num_threads(2)
ds, model = sys.argv[1], sys.argv[2]
seed = 2021
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
args = get_forecast_exp_args(ds, model, 96, 48, 96)
args.root_path = '/home/claude/tfdata/'
args.use_gpu = torch.cuda.is_available(); args.gpu = 0; args.use_multi_gpu = False; args.num_workers = 2
args.checkpoints = '/home/claude/tf_work/checkpoints/'
setting = f"96_48_96/{ds}_{model}_dmodel{args.d_model}_epoch{args.train_epochs}"
t = time.time()
exp = Exp_Fuse_Forecasting(args)
m, vl, tl = exp.train(setting=setting, verbose=False, tqdm_disable=True, save_model=True, raise_fwd_error=True)
out = dict(ds=ds, model=model, val_mse=float(vl), test_mse=float(tl), secs=time.time()-t,
           d_model=args.d_model, epochs=args.train_epochs, lr=args.learning_rate, bs=args.batch_size)
print(json.dumps(out))
with open('/home/claude/tf_work/train_log.jsonl', 'a') as f: f.write(json.dumps(out) + '\n')


In [ ]:
%%writefile /home/claude/tf_work/meta_full.py
# Meta-features for every forecast origin from validation start to end of data (TimeFuse's own extractor)
import os, sys, numpy as np, pandas as pd, time, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '/home/claude/ext/TimeFuse')
from meta_feature import extract_meta_feature
from sklearn.preprocessing import StandardScaler
from multiprocessing import Pool
warnings.simplefilter('ignore')
os.environ['PYTHONWARNINGS']='ignore'
SEQ, PRED = 96, 96
TRAIN_END = 12*30*24
ds = sys.argv[1]
df = pd.read_csv(f'/home/claude/tfdata/{ds}.csv')
X = df.iloc[:, 1:].values
sc = StandardScaler().fit(X[:TRAIN_END]); Z = sc.transform(X)
b1 = TRAIN_END - SEQ
origins = np.arange(TRAIN_END, len(Z) - PRED + 1)  # first target row
def f(o):
    try:
        return extract_meta_feature(Z[o-SEQ:o])
    except Exception:
        return {'__failed__': 1.0}  # constant window; features filled with 0 later, as TimeFuse's Dataset_Meta does for NaN
t = time.time()
with Pool(2) as p: feats = p.map(f, origins, chunksize=64)
M = pd.DataFrame(feats); M.insert(0, 'origin', origins)
M.to_parquet(f'/home/claude/tf_work/meta_{ds}.parquet')
print(ds, M.shape, time.time()-t)


In [ ]:
%%writefile /home/claude/tf_work/infer_full.py
# Run each trained base model on every forecast origin from validation start to the end of the data
import sys, os, numpy as np, pandas as pd, torch, time
sys.path.insert(0, '/home/claude/ext/TimeFuse'); os.chdir('/home/claude/ext/TimeFuse')
from load_configs import get_forecast_exp_args
from exp.exp_fuse_forecasting import Exp_Fuse_Forecasting
from utils.timefeatures import time_features
from sklearn.preprocessing import StandardScaler
torch.set_num_threads(2)
SEQ, LAB, PRED = 96, 48, 96
TRAIN_END = 12*30*24
MODELS = ['DLinear', 'PatchTST', 'TimesNet', 'PAttn', 'TimeMixer', 'TimeXer']
ds = sys.argv[1]
df = pd.read_csv(f'/home/claude/tfdata/{ds}.csv')
X = df.iloc[:, 1:].values
Z = StandardScaler().fit(X[:TRAIN_END]).transform(X).astype(np.float32)
stamp = time_features(pd.to_datetime(df['date'].values), freq='h').transpose(1, 0).astype(np.float32)
origins = np.arange(TRAIN_END, len(Z) - PRED + 1)
trues = np.stack([Z[o:o+PRED] for o in origins])
allp = []
for m in MODELS:
    args = get_forecast_exp_args(ds, m, SEQ, LAB, PRED)
    args.root_path = '/home/claude/tfdata/'; args.use_gpu = False; args.use_multi_gpu = False; args.num_workers = 0
    exp = Exp_Fuse_Forecasting(args)
    ck = f"/home/claude/tf_work/checkpoints/96_48_96/{ds}_{m}_dmodel{args.d_model}_epoch{args.train_epochs}/checkpoint.pth"
    exp.model.load_state_dict(torch.load(ck, map_location='cpu')); exp.model.eval()
    preds = []; t = time.time()
    with torch.no_grad():
        for i in range(0, len(origins), 256):
            ob = origins[i:i+256]
            bx = torch.from_numpy(np.stack([Z[o-SEQ:o] for o in ob]))
            bxm = torch.from_numpy(np.stack([stamp[o-SEQ:o] for o in ob]))
            by = torch.from_numpy(np.stack([Z[o-LAB:o+PRED] for o in ob]))
            bym = torch.from_numpy(np.stack([stamp[o-LAB:o+PRED] for o in ob]))
            dec = torch.cat([by[:, :LAB], torch.zeros_like(by[:, -PRED:])], 1)
            out = exp.model(bx, bxm, dec, bym)[:, -PRED:, :]
            preds.append(out.numpy())
    preds = np.concatenate(preds)
    print(ds, m, preds.shape, 'mse_all', float(((preds-trues)**2).mean()), f'{time.time()-t:.0f}s', flush=True)
    allp.append(preds)
P = np.stack(allp, 1).astype(np.float32)
np.savez_compressed(f'/home/claude/tf_work/preds_{ds}.npz', preds=P, trues=trues, origins=origins, models=np.array(MODELS))


In [ ]:
%%writefile /home/claude/tf_work/analysis.py
# Protocols A, A+, B, C + statistics for the TimeFuse deployment-faithful study
import numpy as np, pandas as pd, torch, torch.nn as nn, json, math
torch.set_num_threads(2)
DS = ['ETTh1', 'ETTh2']; MONTH = 720; PRED = 96
SEEDS = [2021, 2022, 2023, 2024, 2025]
D = {}
for ds in DS:
    z = np.load(f'/home/claude/tf_work/preds_{ds}.npz', allow_pickle=True)
    M = pd.read_parquet(f'/home/claude/tf_work/meta_{ds}.parquet').set_index('origin')
    failed = M.pop('__failed__').notna().sum() if '__failed__' in M else 0
    M = M.loc[z['origins']].fillna(0.0)
    D[ds] = dict(P=z['preds'], Y=z['trues'], o=z['origins'], X=M.values.astype(np.float32),
                 models=list(z['models']), failed=int(failed), feat_names=list(M.columns))
MODELS = D['ETTh1']['models']; K = len(MODELS)
def month(o): return o // MONTH + 1
VAL = lambda o: (o >= 8640) & (o + PRED <= 11520)
TEST = lambda o: (o >= 11520) & (o + PRED <= 14400)
TAIL = lambda o: (o >= 14400) & (o < 17280)   # months 21-24 (the data TSLib never uses)

def fit_fusor(sets, seed, epochs=5, bs=64, lr=5e-4):
    """sets: list of index masks per dataset (same order as DS). Mirrors run_timefuse_exp.ipynb."""
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    Xs = [D[ds]['X'][m] for ds, m in zip(DS, sets)]
    allx = np.concatenate(Xs); mu, sd = allx.mean(0), allx.std(0); sd[sd == 0] = 1
    fc = nn.Linear(Xs[0].shape[1], K); nn.init.kaiming_uniform_(fc.weight, a=math.sqrt(5))
    opt = torch.optim.Adam(fc.parameters(), lr=lr); crit = nn.SmoothL1Loss(beta=0.01)
    data = [(torch.tensor((D[ds]['X'][m] - mu) / sd), torch.tensor(D[ds]['P'][m]), torch.tensor(D[ds]['Y'][m]))
            for ds, m in zip(DS, sets)]
    L = max(len(d[0]) for d in data)
    for ep in range(epochs):
        orders = []
        for d in data:  # length alignment by oversampling, as in get_length_aligned_loaders
            n = len(d[0]); idx = np.concatenate([np.tile(np.arange(n), L // n), rng.choice(n, L % n, replace=False)])
            orders.append(rng.permutation(idx))
        for b in range(0, L, bs):
            for d, od in zip(data, orders):
                ii = od[b:b+bs]
                w = torch.softmax(fc(d[0][ii]), -1)[:, :, None, None]
                loss = crit((w * d[1][ii]).sum(1), d[2][ii])
                opt.zero_grad(); loss.backward(); opt.step()
    return fc, mu, sd

def apply(fc, mu, sd, ds, m):
    with torch.no_grad():
        w = torch.softmax(fc(torch.tensor((D[ds]['X'][m] - mu) / sd)), -1).numpy()
    return (w[:, :, None, None] * D[ds]['P'][m]).sum(1), w

def per_origin_mse(pred, y): return ((pred - y) ** 2).mean((1, 2))
def per_origin_mae(pred, y): return np.abs(pred - y).mean((1, 2))

# ---------- statistics ----------
def block_boot(d, block=168, B=2000, seed=0):
    rng = np.random.default_rng(seed); n = len(d); nb = int(np.ceil(n / block))
    means = np.empty(B)
    for b in range(B):
        starts = rng.integers(0, n - block + 1, nb)
        means[b] = np.concatenate([d[s:s+block] for s in starts])[:n].mean()
    return means
def dm_test(d, h=PRED):
    from scipy import stats
    n = len(d); dbar = d.mean(); dc = d - dbar
    lag = h - 1; gam = [np.dot(dc[k:], dc[:n-k]) / n for k in range(lag + 1)]
    lrv = gam[0] + 2 * sum((1 - k / (lag + 1)) * gam[k] for k in range(1, lag + 1))
    stat = dbar / np.sqrt(lrv / n)
    hln = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    stat *= hln
    p = 2 * stats.t.sf(abs(stat), df=n - 1)
    return float(stat), float(p)
def compare(loss_tf, loss_base, name):
    d = loss_tf - loss_base             # negative = TimeFuse better
    bm = block_boot(d)
    s, p = dm_test(d)
    return dict(comparison=name, mean_diff=float(d.mean()), rel_gain_pct=float(-d.mean() / loss_base.mean() * 100),
                ci_lo=float(np.percentile(bm, 2.5)), ci_hi=float(np.percentile(bm, 97.5)),
                p_not_better=float((bm >= 0).mean()), dm_stat=s, dm_p=p, n=len(d))

R = {'failed_meta_windows': {ds: D[ds]['failed'] for ds in DS}, 'n_meta_features': len(D['ETTh1']['feat_names'])}

# ---------- Protocol A and A+ : fusor fit on validation (paper protocol) ----------
static = [fit_fusor([VAL(D[ds]['o']) for ds in DS], s) for s in SEEDS]
rows, stats_rows, monthly, weights_month = [], [], [], []
for ds in DS:
    o, P, Y = D[ds]['o'], D[ds]['P'], D[ds]['Y']
    v = VAL(o); val_mse = [per_origin_mse(P[v][:, k], Y[v]).mean() for k in range(K)]
    k_val = int(np.argmin(val_mse))
    for split, msk in [('test', TEST(o)), ('tail', TAIL(o))]:
        y = Y[msk]
        base_mse = {MODELS[k]: per_origin_mse(P[msk][:, k], y) for k in range(K)}
        base_mae = {MODELS[k]: per_origin_mae(P[msk][:, k], y) for k in range(K)}
        k_orc = min(base_mse, key=lambda k: base_mse[k].mean())
        mean_pred = P[msk].mean(1)
        tf = [apply(f, mu, sd, ds, msk) for f, mu, sd in static]
        tf_mse = np.mean([per_origin_mse(p, y) for p, _ in tf], 0)
        tf_mae = np.mean([per_origin_mae(p, y) for p, _ in tf], 0)
        seed_mse = [per_origin_mse(p, y).mean() for p, _ in tf]
        row = dict(dataset=ds, split=split, timefuse_mse=float(tf_mse.mean()), timefuse_mse_sd=float(np.std(seed_mse)),
                   timefuse_mae=float(tf_mae.mean()), mean_ens_mse=float(per_origin_mse(mean_pred, y).mean()),
                   val_best=MODELS[k_val], val_best_mse=float(base_mse[MODELS[k_val]].mean()),
                   val_best_mae=float(base_mae[MODELS[k_val]].mean()),
                   oracle_best=k_orc, oracle_best_mse=float(base_mse[k_orc].mean()))
        row.update({f'{m}_mse': float(base_mse[m].mean()) for m in MODELS})
        row.update({f'{m}_mae': float(base_mae[m].mean()) for m in MODELS})
        rows.append(row)
        for nm, base in [('vs val-selected best single', base_mse[MODELS[k_val]]), ('vs oracle best single', base_mse[k_orc]),
                         ('vs simple mean', per_origin_mse(mean_pred, y))]:
            stats_rows.append(dict(dataset=ds, split=split, **compare(tf_mse, base, nm)))
    # monthly breakdown 17..24
    for mth in range(17, 25):
        msk = month(o) == mth
        y = Y[msk]
        base = np.array([per_origin_mse(P[msk][:, k], y).mean() for k in range(K)])
        tfp = [apply(f, mu, sd, ds, msk) for f, mu, sd in static]
        tfm = np.mean([per_origin_mse(p, y).mean() for p, _ in tfp])
        w = np.mean([ww.mean(0) for _, ww in tfp], 0)
        monthly.append(dict(dataset=ds, month=mth, months_after_val=mth - 16, timefuse_static=float(tfm),
                            val_best=float(base[k_val]), oracle_month_best=float(base.min()),
                            oracle_month_best_model=MODELS[int(base.argmin())], mean_ens=float(per_origin_mse(P[msk].mean(1), y).mean()),
                            **{f'w_{m}': float(w[i]) for i, m in enumerate(MODELS)},
                            **{f'mse_{m}': float(base[i]) for i, m in enumerate(MODELS)}))
R['protocol_A'] = rows; R['stats_A'] = stats_rows; R['monthly_static'] = monthly

# ---------- Protocol B: rolling monthly refit (most recent 4 months of labelled windows) ----------
WIN = 4 * MONTH
roll = []
B_loss = {ds: {'tf_roll': [], 'tf_static': [], 'val_best': [], 'roll_best': []} for ds in DS}
for mth in range(17, 25):
    T = (mth - 1) * MONTH
    trmask = lambda o: (o + PRED <= T) & (o + PRED > T - WIN)
    fits = [fit_fusor([trmask(D[ds]['o']) for ds in DS], s) for s in SEEDS]
    for ds in DS:
        o, P, Y = D[ds]['o'], D[ds]['P'], D[ds]['Y']
        tm, em = trmask(o), month(o) == mth
        y = Y[em]
        k_roll = int(np.argmin([per_origin_mse(P[tm][:, k], Y[tm]).mean() for k in range(K)]))
        v = VAL(o); k_val = int(np.argmin([per_origin_mse(P[v][:, k], Y[v]).mean() for k in range(K)]))
        tr = np.mean([per_origin_mse(apply(f, mu, sd, ds, em)[0], y) for f, mu, sd in fits], 0)
        ts = np.mean([per_origin_mse(apply(f, mu, sd, ds, em)[0], y) for f, mu, sd in static], 0)
        vb = per_origin_mse(P[em][:, k_val], y); rb = per_origin_mse(P[em][:, k_roll], y)
        for kk, vv in [('tf_roll', tr), ('tf_static', ts), ('val_best', vb), ('roll_best', rb)]: B_loss[ds][kk].append(vv)
        wr = np.mean([apply(f, mu, sd, ds, em)[1].mean(0) for f, mu, sd in fits], 0)
        roll.append(dict(dataset=ds, month=mth, tf_rolling=float(tr.mean()), tf_static=float(ts.mean()),
                         val_best=float(vb.mean()), rolling_best_single=float(rb.mean()), rolling_best_model=MODELS[k_roll],
                         **{f'wroll_{m}': float(wr[i]) for i, m in enumerate(MODELS)}))
R['protocol_B'] = roll
statsB = []
for ds in DS:
    for split, sl in [('test', slice(0, 4)), ('tail', slice(4, 8))]:
        L = {k: np.concatenate(v[sl]) for k, v in B_loss[ds].items()}
        statsB.append(dict(dataset=ds, split=split, **compare(L['tf_roll'], L['tf_static'], 'rolling TF vs static TF')))
        statsB.append(dict(dataset=ds, split=split, **compare(L['tf_roll'], L['roll_best'], 'rolling TF vs rolling best single')))
        statsB.append(dict(dataset=ds, split=split, **compare(L['tf_static'], L['val_best'], 'static TF vs val-selected best single')))
R['stats_B'] = statsB

# ---------- Protocol C: staleness curve (fusor window ends k months before eval month; months 21-24) ----------
stale = []
for mth in range(21, 25):
    T = (mth - 1) * MONTH
    for k in range(0, 5):
        end = T - k * MONTH
        trm = lambda o: (o + PRED <= end) & (o + PRED > end - WIN) & (o >= 8640)
        fits = [fit_fusor([trm(D[ds]['o']) for ds in DS], s) for s in SEEDS]
        for ds in DS:
            o, P, Y = D[ds]['o'], D[ds]['P'], D[ds]['Y']; em = month(o) == mth; y = Y[em]
            tfm = np.mean([per_origin_mse(apply(f, mu, sd, ds, em)[0], y).mean() for f, mu, sd in fits])
            tmk = trm(o); kb = int(np.argmin([per_origin_mse(P[tmk][:, j], Y[tmk]).mean() for j in range(K)]))
            stale.append(dict(dataset=ds, month=mth, lag_months=k, timefuse=float(tfm),
                              sel_best_single=float(per_origin_mse(P[em][:, kb], y).mean()), n_fit=int(tmk.sum())))
R['protocol_C'] = stale
json.dump(R, open('/home/claude/tf_work/results.json', 'w'), indent=1)
print('done')


In [ ]:
%%writefile /home/claude/tf_work/figs.py
import json, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
R = json.load(open('/home/claude/tf_work/results.json'))
C = dict(blue='#2a78d6', orange='#eb6834', aqua='#1baf7a', yellow='#eda100', magenta='#e87ba4', green='#008300')
TXT, MUTED, GRID = '#0b0b0b', '#52514e', '#e4e3df'
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8.5, 'axes.edgecolor': MUTED, 'axes.labelcolor': TXT,
                     'xtick.color': MUTED, 'ytick.color': MUTED, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6, 'axes.axisbelow': True})
ms = pd.DataFrame(R['monthly_static']); rb = pd.DataFrame(R['protocol_B'])
# Fig 1: relative gain vs val-selected best single by month
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.5), sharey=True)
for ax, ds in zip(axes, ['ETTh1', 'ETTh2']):
    a = ms[ms.dataset == ds].sort_values('month'); b = rb[rb.dataset == ds].sort_values('month')
    x = a.months_after_val.values
    g_static = (a.val_best - a.timefuse_static) / a.val_best * 100
    g_roll = (b.val_best.values - b.tf_rolling.values) / b.val_best.values * 100
    ax.axhline(0, color=MUTED, lw=0.8)
    ax.axvspan(4.5, 8.5, color='#f3f2ee', zorder=0)
    ax.plot(x, g_static, '-o', color=C['blue'], lw=2, ms=4.5, label='TimeFuse, fusor fit once on validation')
    ax.plot(x, g_roll, '-s', color=C['orange'], lw=2, ms=4.5, label='TimeFuse, fusor refit monthly')
    ax.set_title(ds, color=TXT, fontsize=9, loc='left'); ax.set_xticks(range(1, 9)); ax.set_xlim(0.5, 8.5)
    ax.set_xlabel('Months after the validation window')
    ax.text(1, ax.get_ylim()[1] if False else 0, '', color=MUTED)
axes[0].set_ylabel('MSE gain over val-selected\nbest single model (%)')
for ax in axes:
    yl = ax.get_ylim(); ax.text(2.5, yl[1] - (yl[1]-yl[0])*0.06, 'paper test window', ha='center', color=MUTED, fontsize=7.5)
    ax.text(6.5, yl[1] - (yl[1]-yl[0])*0.06, 'unused months 21-24', ha='center', color=MUTED, fontsize=7.5)
h, l = axes[0].get_legend_handles_labels(); fig.legend(h, l, loc='lower center', ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.04))
fig.tight_layout(rect=(0, 0.08, 1, 1)); fig.savefig('/home/claude/tf_work/fig1.png', dpi=220, bbox_inches='tight'); plt.close(fig)
# Fig 2: staleness curve
st = pd.DataFrame(R['protocol_C'])
fig, ax = plt.subplots(figsize=(3.4, 2.4))
for ds, col in [('ETTh1', C['blue']), ('ETTh2', C['orange'])]:
    s = st[st.dataset == ds]
    rel = s.groupby('lag_months').apply(lambda d: np.mean(d.timefuse.values / s[(s.lag_months == 0)].set_index('month').loc[d.month, 'timefuse'].values) * 100 - 100)
    ax.plot(rel.index, rel.values, '-o', color=col, lw=2, ms=4.5, label=ds)
    ax.annotate(ds, (rel.index[-1], rel.values[-1]), xytext=(4, 0), textcoords='offset points', color=TXT, fontsize=7.5, va='center')
ax.axhline(0, color=MUTED, lw=0.8); ax.set_xticks(range(0, 5)); ax.set_xlim(-0.3, 4.8)
ax.set_xlabel('Age of the fusor\'s training window (months)'); ax.set_ylabel('MSE change vs fresh fusor (%)')
ax.legend(frameon=False, loc='upper left'); fig.tight_layout(); fig.savefig('/home/claude/tf_work/fig2.png', dpi=220, bbox_inches='tight'); plt.close(fig)
# Fig 3: weight drift (static fusor) + monthly best model
models = ['DLinear', 'PatchTST', 'TimesNet', 'PAttn', 'TimeMixer', 'TimeXer']; cols = list(C.values())
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.5), sharey=True)
for ax, ds in zip(axes, ['ETTh1', 'ETTh2']):
    b = rb[rb.dataset == ds].sort_values('month'); x = b.month.values - 16
    W = np.array([b[f'wroll_{m}'].values for m in models])
    ax.stackplot(x, W, colors=cols, labels=models, edgecolor='white', linewidth=1.0)
    a = ms[ms.dataset == ds].sort_values('month')
    for xi, bm in zip(a.months_after_val, a.oracle_month_best_model):
        ax.text(xi, 1.02, bm.replace('Mixer', 'Mix.').replace('Net', 'N.'), rotation=60, fontsize=6.3, color=MUTED, ha='left', va='bottom')
    ax.set_xlim(1, 8); ax.set_ylim(0, 1); ax.set_xticks(range(1, 9)); ax.set_title(ds, loc='left', fontsize=9, color=TXT, pad=34)
    ax.set_xlabel('Months after the validation window'); ax.grid(False)
axes[0].set_ylabel('Mean fusor weight\n(monthly refit)')
h, l = axes[0].get_legend_handles_labels(); fig.legend(h, l, loc='lower center', ncol=6, frameon=False, bbox_to_anchor=(0.5, -0.05))
fig.tight_layout(rect=(0, 0.08, 1, 1)); fig.savefig('/home/claude/tf_work/fig3.png', dpi=220, bbox_inches='tight'); plt.close(fig)
print('figs ok')


In [ ]:
# 1) base models: 6 models x 2 datasets, official TSLib configs used by TimeFuse (seed 2021)
!cd /home/claude/tf_work && for ds in ETTh1 ETTh2; do for m in DLinear PatchTST TimesNet PAttn TimeMixer TimeXer; do python -W ignore train_base.py $ds $m 2>&1 | grep -v "Updating learning" | tail -1; done; done

In [ ]:
# 2) meta-features for every forecast origin (CPU, ~15 min)
!cd /home/claude/tf_work && python -W ignore meta_full.py ETTh1 2>&1 | grep -a "^ETTh" ; python -W ignore meta_full.py ETTh2 2>&1 | grep -a "^ETTh"

In [ ]:
# 3) base-model predictions from validation start to the end of the data
!cd /home/claude/tf_work && python -W ignore infer_full.py ETTh1 && python -W ignore infer_full.py ETTh2

In [ ]:
# 4) protocols A, A+, B, C + statistics, then figures
!cd /home/claude/tf_work && python -W ignore analysis.py && python figs.py

In [ ]:
!cd /home/claude/tf_work && zip -q timefuse_results.zip results.json train_log.jsonl fig1.png fig2.png fig3.png && ls -la timefuse_results.zip
from google.colab import files; files.download('/home/claude/tf_work/timefuse_results.zip')